# 🧭 Life OS — a private, multi-agent AI for your real life (finance first)
> **Kaggle AI Agents: Intensive Vibe Coding Capstone — Concierge Agents track.**

## The problem
Personal-finance tools are *generic* — they don't know your goals, your risk appetite, your non-negotiables, or that you and your partner decide big things together. So their advice is generic too.

## The solution
**Life OS** is a private mesh of specialist agents that ingest your real financial data, compute the exact numbers, and reason about your decisions **grounded in your own documented values** — interrupting you only when a human decision matters.

| Agent | Role |
|---|---|
| **Archivist** | Ingests dropped documents → text → vector memory (RAG) |
| **Accountant** | Parses statements, categorizes, computes **exact** finance math |
| **Mentor** | Conversational Q&A grounded in your numbers + values |
| **Board of Directors** | 5-agent debate that rules on major decisions (centerpiece) |
| **Daily Guide** | Proactive daily safe-to-spend + one nudge |
| **Signal / Expense agents** | Windfall catalyst alerts · ambient expense triage (ADK, Cloud Run) |

## Course concepts demonstrated
| Concept | Where |
|---|---|
| Multi-agent system (ADK) | Board of Directors + ambient Expense Sentinel |
| Agent tools / memory | foundation document + finance tools |
| Security & guardrails | deterministic PII/injection screen; windfall cap in code; synthetic-data privacy |
| Deployability | Expense Sentinel deployed to Cloud Run |
| Antigravity + Agents CLI | built in Antigravity; deployed via agents-cli |

## ⚙️ Setup
On Kaggle this clones the public repo and loads the Gemini key from Kaggle Secrets. Locally, run it from the repo root.

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Make the `src` package importable (Kaggle: clone the public repo; local: use cwd).
if Path('src').exists():
    sys.path.insert(0, '.')
else:
    if not Path('life-os-agents').exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/shravan-yampati/life-os-agents.git'], check=False)
    os.chdir('life-os-agents'); sys.path.insert(0, '.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'google-genai', 'pydantic-settings', 'numpy', 'pypdf'], check=False)

# Gemini API key (Kaggle Secrets), then point the provider at real Gemini.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['GOOGLE_API_KEY'] = UserSecretsClient().get_secret('GOOGLE_API_KEY')
except Exception:
    pass
os.environ['CLOUD_PROVIDER'] = 'gcp' if os.environ.get('GOOGLE_API_KEY') else 'local'
print('Provider:', os.environ['CLOUD_PROVIDER'])

## 🔒 A synthetic persona (privacy by design)
The system runs on *your real data* privately. For this public demo we use a **synthetic persona** — fake but realistic. (We never overwrite a real foundation if one already exists.) For the **Concierge** track, keeping personal data out of the public artifact is the point.

In [ ]:
SYNTHETIC_FOUNDATION = '''# Life OS Foundation (SYNTHETIC demo persona)
## Profile
Alex Rivera — software engineer, married; big financial decisions are made together.
Values working smart over grinding (leverage, automation).
## Goals
- Save $80,000 for a house down payment by end of next year (all income engines combined).
- The salary is the engine; current speculative capital is only ~$1,000.
## Risk posture (three buckets)
1. Core/house savings — sacred, never touched.
2. $1K growth bucket — calculated, clever-not-YOLO bets.
3. Windfall bucket — UNEXPECTED money only; high-risk options OK, never from core savings.
## Non-negotiables (never debated away)
Keep an emergency fund; never raid house savings for a trade; no new high-interest debt;
$1K is the max speculative risk right now.
## Communication tone
Direct, tough-love, specific numbers over vague encouragement.
'''

fpath = Path('brainstorms/life-os-foundation.md')
fpath.parent.mkdir(exist_ok=True)
if not fpath.exists():
    fpath.write_text(SYNTHETIC_FOUNDATION, encoding='utf-8')
    print('Synthetic persona written.')
else:
    print('A foundation already exists — leaving it untouched.')

## 1️⃣ Accountant — exact finance (never "RAG" a number)
Money is parsed and computed with `Decimal` in code. The LLM only ever *narrates* these exact figures.

In [ ]:
from src.finance.statements import load_transactions
from src.finance.categorize import categorize
from src.finance.analyze import analyze

txns = categorize(load_transactions('data/finance/sample_transactions.csv'))
summary = analyze(txns)
print(f'Income:  ${summary.total_income:,.2f}')
print(f'Spend:   ${summary.total_spend:,.2f}')
print(f'Net:     ${summary.net:,.2f}  (savings rate {summary.savings_rate}%)')
print('Top categories:', dict(list(summary.spend_by_category.items())[:3]))

## 2️⃣ Mentor — grounded chat
Ask anything about the money; answers are grounded in the exact numbers above + the persona's values.

In [ ]:
from src.agents.mentor import Mentor
print(Mentor(person='me').ask('How am I doing on savings, and where am I overspending?').answer)

## 3️⃣ Board of Directors — the centerpiece 🏛️
Four persona advisors debate a decision; a Synthesizer weighs the debate against the persona's foundation and **refuses anything that breaks a non-negotiable.**

In [ ]:
from src.agents.board import convene_board
ruling = convene_board('Should I put my $1,000 into a single high-risk options trade to grow it fast?')
for op in ruling.opinions:
    print(f'-- {op.persona} --\n{op.text}\n')
print('=== SYNTHESIZER RULING ===')
print(ruling.verdict)

## 4️⃣ Daily Guide — a proactive daily note
Computes today's safe-to-spend + flags in code, then writes a short message in the persona's tone.

In [ ]:
from src.agents.daily_guide import generate_daily_message
msg = generate_daily_message(person='me')
print('FACTS:\n' + msg.facts + '\n')
print('MESSAGE:\n' + msg.message)

## 🛡️ Security, guardrails & privacy (Concierge track)
- **Deterministic security screen** (PII redaction + prompt-injection checks) runs *before* the LLM in the ambient Expense Sentinel.
- **Hard-coded guardrails:** the Signal Agent caps any suggested play to the windfall budget *in code*, and the Board can never override a non-negotiable.
- **Privacy by design:** real financial data and values stay local and gitignored; this public notebook runs on a synthetic persona. Secrets load from Kaggle Secrets / Secret Manager, never hardcoded.

## Summary
Life OS is a coherent multi-agent system — exact finance math, grounded chat, a values-enforcing debate, a proactive daily guide, plus an ambient ADK expense agent deployed to Cloud Run. Built one domain at a time (finance first), designed to extend to health and career.

**Code:** https://github.com/shravan-yampati/life-os-agents · **Built with:** Google Gemini · Google ADK · Cloud Run · FastAPI · pgvector · Antigravity.